This notebook attempts to detemine what variables require weighting, and how to apply the weighting.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pygmt
from pygmt_helper import plotting

import nn_gmm as nng

In [ ]:
imdb_ffp = Path("/Users/claudy/dev/work/data/nn_gmm/20251222_CS200_DSSlab_rotd50_imdb.duckdb")
test_events_ffps = Path("/Users/claudy/dev/work/data/nn_gmm/test_events.npy")

In [ ]:
with nng.DuckIMDB(imdb_ffp, readonly=True) as imdb:
    site_df = imdb.get_site_df(max_grid_level=0, min_grid_level=0)
    sites = site_df.site_id.values.astype(str)

    event_df = imdb.get_event_df()
    rel_df = imdb.get_rel_df()


# Drop test events
test_events = np.load(test_events_ffps).astype(str)
event_df = event_df[~event_df.event_id.isin(test_events)]

record_info_df = imdb.get_record_info_df(sites=sites, events=event_df.event_id.values.astype(str))
site_event_data = imdb.get_site_event_df(sites=site_df.site_id.values.astype(str))
    
print("Event DF shape:", event_df.shape)

In [ ]:
# Add site-event int id
record_info_df["site_event_int_id"] = nng.utils.get_site_event_int_id(record_info_df.site_int_id.values, record_info_df.event_int_id.values)

# Add magnitude, rrup, vs30, and tectonic type
record_info_df["vs30"] = site_df.loc[record_info_df.site_int_id, "vs30"].values
record_info_df["magnitude"] = event_df.loc[record_info_df.event_int_id, "magnitude"].values
record_info_df["rrup"] = site_event_data.loc[record_info_df.site_event_int_id, "rrup"].values
record_info_df["tect_type"] = event_df.loc[record_info_df.event_int_id, "tect_type"].values
record_info_df["depth"] = rel_df.loc[record_info_df.rel_int_id, "hypo_depth"].values

## Magnitude

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

ax.hist(record_info_df.magnitude, bins=30, edgecolor="black")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.xaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax.set_xlabel("Magnitude, $M_W$")
ax.set_ylabel("Number of records")
ax.set_title("Magnitude Distribution")
fig.tight_layout()

In [ ]:
mag_bins = nng.constants.MAG_WEIGHTING_BINS
mag_bin_names = nng.constants.MAG_WEIGHTING_BIN_NAMES

record_info_df["mag_bin"] = pd.cut(record_info_df.magnitude, mag_bins, labels=mag_bin_names)
mag_bin_counts = record_info_df.mag_bin.value_counts().sort_index()

In [ ]:
mag_max_weight = 2.5
mag_bin_weights = np.clip((mag_bin_counts.max() / mag_bin_counts) - 1, 0.0, mag_max_weight) 
record_info_df["mag_bin_weight"] = record_info_df.mag_bin.map(mag_bin_weights).astype(float)
print("Using max mag weight:", mag_max_weight)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

ax1.hist(record_info_df.magnitude, bins=mag_bins, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(0.25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Magnitude Bin")

# ax2.hist(record_info_df.magnitude, bins=mag_bins, weights=record_info_df["mag_bin_weight"], edgecolor="black")
ax2.bar(mag_bin_names, mag_bin_weights.values, edgecolor="black")
ax2.set_title("Additional Magnitude Weights")
ax2.set_xlabel("Magnitude Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_minor_locator(plt.MultipleLocator(0.25))

ax3.hist(record_info_df.magnitude, bins=mag_bins, weights=1 + record_info_df["mag_bin_weight"], edgecolor="black")
ax3.set_title("Base + Magnitude Weight")
ax3.set_xlabel("Magnitude Bin")
ax3.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax3.xaxis.set_minor_locator(plt.MultipleLocator(0.25))

fig.tight_layout()

## $R_{Rup}$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.rrup, bins=10, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("$R_{Rup}$ Distribution")

log_rrup_bins = np.logspace(np.log(1), np.log(300), 10, base=np.e)
ax2.hist(record_info_df.rrup, bins=log_rrup_bins, edgecolor="black")
ax2.set_title("$R_{Rup}$ Distribution (Log Scale)")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xscale("log")
# ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))

fig.tight_layout()

In [ ]:
rrup_bins = nng.constants.RRUP_WEIGHTING_BINS
rrup_bin_names = nng.constants.RRUP_WEIGHTING_BIN_NAMES

record_info_df["rrup_bin"] = pd.cut(record_info_df.rrup, rrup_bins, labels=rrup_bin_names)

rrup_bin_counts = record_info_df.rrup_bin.value_counts().sort_index()

In [ ]:
max_rrup_weight = 2.5
rrup_bin_weights = np.clip((rrup_bin_counts.max() / rrup_bin_counts) - 1, 0.0, max_rrup_weight) 
record_info_df["rrup_bin_weight"] = record_info_df.rrup_bin.map(rrup_bin_weights).astype(float)
print("Using max rrup weight:", max_rrup_weight)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

ax1.hist(record_info_df.rrup, bins=rrup_bins, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Rrup Bin")

# ax2.hist(record_info_df.rrup, bins=rrup_bins, weights=record_info_df["rrup_bin_weight"], edgecolor="black")
ax2.bar(rrup_bin_names, rrup_bin_weights.values, edgecolor="black")
ax2.set_title("Additional Rrup Weights")
ax2.set_xlabel("Rrup Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")

ax3.hist(record_info_df.rrup, bins=rrup_bins, weights=1 + record_info_df["rrup_bin_weight"], edgecolor="black")
ax3.set_title("Base + Rrup Weight")
ax3.set_xlabel("Rrup Bin")
ax3.grid(linewidth=0.5, alpha=0.5, linestyle="--")


fig.tight_layout()

## $V_{S30}$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.hist(record_info_df.vs30, bins=10, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("$Vs30$ Distribution")

log_vs30_bins = nng.constants.VS30_WEIGHTING_BINS
ax2.hist(record_info_df.vs30, bins=log_vs30_bins, edgecolor="black")
ax2.set_title("$Vs30$ Distribution (Log Scale)")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))
# ax2.set_xscale("log")

fig.tight_layout()

This one is the trickiest as there there is significant more sites in the 400-500 range than any other Vs30 values.

In [ ]:
vs30_bins = nng.constants.VS30_WEIGHTING_BINS
vs30_bin_names = nng.constants.VS30_WEIGHTING_BIN_NAMES

record_info_df["vs30_bin"] = pd.cut(record_info_df.vs30, vs30_bins, labels=vs30_bin_names)

vs30_bin_counts = record_info_df.vs30_bin.value_counts().sort_index()

In [ ]:
max_vs30_weight = 2.5
vs30_bin_weights = np.clip((vs30_bin_counts.max() / vs30_bin_counts) - 1, 0.0, max_vs30_weight)
record_info_df["vs30_bin_weight"] = record_info_df.vs30_bin.map(vs30_bin_weights)
print("Using max vs30 weight:", max_vs30_weight)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

ax1.hist(record_info_df.vs30, bins=vs30_bins, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Vs30 Bin")
ax1.set_xlim(vs30_bins[0], vs30_bins[-1])

ax2.bar(vs30_bin_names, vs30_bin_weights.values, edgecolor="black")
ax2.set_title("Additional Vs30 Weights")
ax2.set_xlabel("Vs30 Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.xaxis.set_minor_locator(plt.MultipleLocator(25))

ax3.hist(record_info_df.vs30, bins=vs30_bins, weights=1 + record_info_df["vs30_bin_weight"], edgecolor="black")
ax3.set_title("Base + Vs30 Weight")
ax3.set_xlabel("Vs30 Bin")
ax3.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax3.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax3.set_xlim(vs30_bins[0], vs30_bins[-1])

fig.tight_layout()

## Tectonic Type

In [ ]:
tect_type_counts = record_info_df.tect_type.value_counts()

In [ ]:
max_tect_type_weight = 2.5
tect_type_weights = np.clip((tect_type_counts.max() / tect_type_counts) - 1, 0.0, max_tect_type_weight)
record_info_df["tect_type_weight"] = record_info_df.tect_type.map(tect_type_weights)
print("Using max tectonic type weight:", max_tect_type_weight)

In [ ]:
record_info_df["base_and_tect_weight"] = 1 + record_info_df["tect_type_weight"]
weighted_counts = record_info_df.groupby("tect_type", observed=True)["base_and_tect_weight"].sum()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

ax1.bar(tect_type_counts.index, tect_type_counts.values, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Tectonic Type")

ax2.bar(tect_type_weights.index, tect_type_weights.values, edgecolor="black")
ax2.set_title("Additional Tectonic Type Weights")
ax2.set_xlabel("Tectonic Type")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")

ax3.bar(weighted_counts.index, weighted_counts.values, edgecolor="black")
ax3.set_title("Base + Tectonic Type Weight")
ax3.set_xlabel("Tectonic Type")
ax3.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()

## Depth

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(16, 6))

ax1.hist(record_info_df.depth, bins=nng.constants.DEPTH_WEIGHTING_BINS, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
# ax1.xaxis.set_minor_locator(plt.MultipleLocator(25))
ax1.set_title("Depth Distribution")

fig.tight_layout()

In [ ]:
record_info_df["depth_bin"] = pd.cut(record_info_df.depth, nng.constants.DEPTH_WEIGHTING_BINS, labels=nng.constants.DEPTH_WEIGHTING_BIN_NAMES)
depth_bin_counts = record_info_df.depth_bin.value_counts().sort_index()

max_depth_weight = 2.5
depth_bin_weights = np.clip((depth_bin_counts.max() / depth_bin_counts) - 1, 0.0, max_depth_weight)
record_info_df["depth_bin_weight"] = record_info_df.depth_bin.map(depth_bin_weights)
print("Using max depth weight:", max_depth_weight)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

ax1.hist(record_info_df.depth, bins=nng.constants.DEPTH_WEIGHTING_BINS, edgecolor="black")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_title("Original")
ax1.set_ylabel("Count")
ax1.set_xlabel("Depth Bin")

ax2.bar(nng.constants.DEPTH_WEIGHTING_BIN_NAMES, depth_bin_weights.values, edgecolor="black")
ax2.set_title("Additional Depth Weights")
ax2.set_xlabel("Depth Bin")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")

ax3.hist(record_info_df.depth, bins=nng.constants.DEPTH_WEIGHTING_BINS, weights=1 + record_info_df["depth_bin_weight"], edgecolor="black")
ax3.set_title("Base + Depth Weight")
ax3.set_xlabel("Depth Bin")
ax3.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()